# 01 — API de Anthropic (Claude) — Notebook interactivo

> **Bloque:** APIs de IA | **Nivel:** Práctico
>
> Complementario al tutorial [01-api-anthropic-claude.md](../../apis/01-api-anthropic-claude.md)

Este notebook te permite ejecutar y modificar todos los ejemplos del tutorial directamente.

In [ ]:
# %pip install anthropic python-dotenv

import anthropic
import os
import json
import base64
from pathlib import Path

# Configura tu API key
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'

client = anthropic.Anthropic()
print('✓ Cliente Anthropic inicializado')

## 1. Primera llamada a la API

In [ ]:
message = client.messages.create(
    model='claude-haiku-4-5-20251001',
    max_tokens=512,
    messages=[{
        'role': 'user',
        'content': 'Explica qué es la inteligencia artificial en exactamente 3 frases.'
    }]
)

print('=== RESPUESTA ===')
print(message.content[0].text)
print(f'\n=== USO ===')
print(f'Tokens entrada:  {message.usage.input_tokens}')
print(f'Tokens salida:   {message.usage.output_tokens}')
print(f'Coste estimado:  ${(message.usage.input_tokens * 0.00000025 + message.usage.output_tokens * 0.00000125):.6f}')

## 2. Conversación con historial

In [ ]:
historial = []

def chat(mensaje: str, modelo: str = 'claude-haiku-4-5-20251001') -> str:
    historial.append({'role': 'user', 'content': mensaje})
    respuesta = client.messages.create(
        model=modelo,
        max_tokens=512,
        messages=historial
    )
    texto = respuesta.content[0].text
    historial.append({'role': 'assistant', 'content': texto})
    return texto

# Simulamos una conversación
r1 = chat('¿Cuáles son los 3 modelos de IA más usados en empresas hoy?')
print('Respuesta 1:', r1)

r2 = chat('¿Cuál de los que mencionaste sería mejor para analizar contratos legales?')
print('\nRespuesta 2:', r2)

print(f'\nMensajes en el historial: {len(historial)}')

## 3. System prompt

In [ ]:
system_prompts = {
    'sin_system': None,
    'formal': 'Responde siempre en un tono muy formal y académico. Usa vocabulario técnico elevado.',
    'emoji': 'Responde de forma muy informal e incluye emojis relevantes en cada frase 😊',
    'conciso': 'Responde SIEMPRE en exactamente una frase. Sin excepciones.',
}

pregunta = '¿Qué es ChatGPT?'

for nombre, system in system_prompts.items():
    kwargs = dict(
        model='claude-haiku-4-5-20251001',
        max_tokens=200,
        messages=[{'role': 'user', 'content': pregunta}]
    )
    if system:
        kwargs['system'] = system
    
    r = client.messages.create(**kwargs)
    print(f'\n[{nombre}]')
    print(r.content[0].text)

## 4. Streaming

In [ ]:
from IPython.display import clear_output, display
import sys

print('Streaming de respuesta (los tokens aparecen uno a uno):\n')
print('-' * 50)

texto_acumulado = ''
with client.messages.stream(
    model='claude-haiku-4-5-20251001',
    max_tokens=300,
    messages=[{'role': 'user', 'content': 'Escribe un párrafo sobre el futuro de la IA en 2030.'}]
) as stream:
    for texto in stream.text_stream:
        texto_acumulado += texto
        print(texto, end='', flush=True)

print('\n' + '-' * 50)
print(f'Total: {len(texto_acumulado)} caracteres')

## 5. Parámetros: temperatura y max_tokens

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Visualizar el efecto de max_tokens en el coste
tokens_entrada = 500
precio_entrada_mtok = 0.25   # Haiku: $0.25 por MTok entrada
precio_salida_mtok = 1.25    # Haiku: $1.25 por MTok salida

max_tokens_opciones = [128, 256, 512, 1024, 2048, 4096]
costes = [
    (tokens_entrada * precio_entrada_mtok + mt * precio_salida_mtok) / 1_000_000
    for mt in max_tokens_opciones
]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar([str(mt) for mt in max_tokens_opciones], 
              [c * 1000 for c in costes],  # en milicéntimos
              color='#3498DB', alpha=0.8, edgecolor='black')
ax.set_xlabel('max_tokens configurado')
ax.set_ylabel('Coste máximo por llamada (m€)')
ax.set_title(f'Coste máximo según max_tokens (Claude Haiku, {tokens_entrada} tokens entrada)')
ax.bar_label(bars, fmt='%.3f m€', padding=3)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()
print('Consejo: ajusta max_tokens al mínimo necesario para controlar costes.')

## 6. Tool Use (uso de herramientas)

In [ ]:
# Definición de herramientas
tools = [
    {
        'name': 'calcular',
        'description': 'Realiza operaciones matemáticas básicas',
        'input_schema': {
            'type': 'object',
            'properties': {
                'operacion': {'type': 'string', 'enum': ['suma', 'resta', 'multiplicacion', 'division']},
                'a': {'type': 'number'},
                'b': {'type': 'number'}
            },
            'required': ['operacion', 'a', 'b']
        }
    }
]

# Función real
def calcular(operacion: str, a: float, b: float) -> dict:
    ops = {'suma': a + b, 'resta': a - b, 'multiplicacion': a * b, 'division': a / b if b != 0 else None}
    return {'resultado': ops.get(operacion), 'operacion': f'{a} {operacion} {b}'}

# Primera llamada
respuesta = client.messages.create(
    model='claude-haiku-4-5-20251001',
    max_tokens=300,
    tools=tools,
    messages=[{'role': 'user', 'content': '¿Cuánto es 1.234 multiplicado por 567?'}]
)

print(f'Stop reason: {respuesta.stop_reason}')

if respuesta.stop_reason == 'tool_use':
    tool_use = next(b for b in respuesta.content if b.type == 'tool_use')
    print(f'Herramienta solicitada: {tool_use.name}')
    print(f'Argumentos: {tool_use.input}')
    
    # Ejecutar la herramienta
    resultado = calcular(**tool_use.input)
    print(f'Resultado local: {resultado}')
    
    # Segunda llamada
    respuesta_final = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=200,
        tools=tools,
        messages=[
            {'role': 'user', 'content': '¿Cuánto es 1.234 multiplicado por 567?'},
            {'role': 'assistant', 'content': respuesta.content},
            {'role': 'user', 'content': [{
                'type': 'tool_result',
                'tool_use_id': tool_use.id,
                'content': json.dumps(resultado)
            }]}
        ]
    )
    print(f'\nRespuesta final: {respuesta_final.content[0].text}')

---
**Tutorial relacionado:** [01-api-anthropic-claude.md](../../apis/01-api-anthropic-claude.md)